In [ ]:
#Import all libraries
import numpy as np
import pandas as pd
import nltk
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from wordcloud import WordCloud, STOPWORDS
import string
import seaborn as sns
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import warnings
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix , classification_report
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/PFA-2A/dataset/spam_ham_dataset.csv"
df = pd.read_csv(path)
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df['label'].value_counts()

In [ ]:
df = df.drop('Unnamed: 0', axis=1)
df = df.drop('label', axis=1)



In [ ]:
print(f"Data with duplicates shape : {df.shape}")
df = df.drop_duplicates()
print(f"Data without duplicates shape : {df.shape}")

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
#Creating a new feature, extracting subject of each email
subjects = []
for i in range(len(df)):
    ln = df["text"].iloc[i]
    line = ""
    for i in ln:
        if(i == '\r'):
            break
        line = line + i
    line = line.replace("Subject" , "")
    subjects.append(line)

df['Subject'] = subjects

In [ ]:
df.columns = ["Email_text" , "Labels" , "Email_Subject"]

In [ ]:
#This function swaps 2 columns inside the dataframe
def swap_columns(df, col1, col2):
    col_list = list(df.columns)
    x, y = col_list.index(col1), col_list.index(col2)
    col_list[y], col_list[x] = col_list[x], col_list[y]
    df = df[col_list]
    return df

In [ ]:
df = swap_columns(df, 'Labels', 'Email_Subject')

In [ ]:
df['Email_Subject'] = df['Email_Subject'].str.lower()
df['Email_text'] = df['Email_text'].str.lower()

In [ ]:
spam_df = df[df["Labels"] == 1]
ham_df = df[df["Labels"] == 0]

In [ ]:
spam_df.head()

In [ ]:
ham_df.head()

In [ ]:
#creating spam subject corpus
Subject_corpus_spam = ""
for i in range(len(spam_df)):
    Subject_corpus_spam = Subject_corpus_spam + spam_df['Email_Subject'].iloc[i]

In [ ]:
#creating spam text corpus
Text_corpus_spam = ""
for i in range(len(spam_df)):
    Text_corpus_spam = Text_corpus_spam + spam_df['Email_text'].iloc[i]

In [ ]:
#creating non-spam subject corpus
Subject_corpus_non_spam = ""
for i in range(len(ham_df)):
    Subject_corpus_non_spam = Subject_corpus_non_spam + ham_df['Email_Subject'].iloc[i]

In [ ]:
#creating non-spam text corpus
Text_corpus_non_spam = ""
for i in range(len(ham_df)):
    Text_corpus_non_spam = Text_corpus_non_spam + ham_df['Email_text'].iloc[i]

In [ ]:
#creating word cloud for given corpus
def make_wordcloud(corpus , stopword):
    word_cloud = WordCloud(
            width=3000,
            height=2000,
            random_state=1,
            background_color="black",
            colormap="Pastel1",
            collocations=False,
            stopwords=stopword,
            ).generate(corpus)
    return word_cloud

In [ ]:
#plotting word cloud for Spam Subject corpus
Spam_Subject_wordcloud = make_wordcloud (Subject_corpus_spam , STOPWORDS)
plt.figure(figsize=(13, 13))
plt.title("Most common words in Spam email Subjects", fontdict={'size': 20, 'color': 'black',
                                  'verticalalignment': 'bottom'})
plt.imshow(Spam_Subject_wordcloud)
plt.axis("off")
plt.show()

In [ ]:
#plotting word cloud for Spam Text corpus
Spam_Text_wordcloud = make_wordcloud (Text_corpus_spam , STOPWORDS)
plt.figure(figsize=(13, 13))
plt.title("Most common words in Spam email", fontdict={'size': 20, 'color': 'black',
                                  'verticalalignment': 'bottom'})
plt.imshow(Spam_Text_wordcloud)
plt.axis("off")
plt.show()

In [ ]:
#plotting word cloud for Non Spam Subject corpus
Non_Spam_Subject_wordcloud = make_wordcloud (Subject_corpus_non_spam , STOPWORDS)
plt.figure(figsize=(13, 13))
plt.title("Most common words in NON-Spam email subject", fontdict={'size': 20, 'color': 'black',
                                  'verticalalignment': 'bottom'})
plt.imshow(Non_Spam_Subject_wordcloud)
plt.axis("off")
plt.show()

In [ ]:
#plotting word cloud for Non Spam Text corpus
Non_Spam_Subject_wordcloud = make_wordcloud (Text_corpus_non_spam , STOPWORDS)
plt.figure(figsize=(13, 13))
plt.title("Most common words in NON-Spam email", fontdict={'size': 20, 'color': 'black',
                                  'verticalalignment': 'bottom'})
plt.imshow(Non_Spam_Subject_wordcloud)
plt.axis("off")
plt.show()

In [ ]:
#create a corpus containing all the email text
Email_text_corpus = []
for line in df['Email_text']:
    words = line.split(" ")
    Email_text_corpus.append(words)

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

def preprocess_text(text):

    text = str(text).lower()

    # garder lettres + chiffres (IMPORTANT)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # supprimer espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))

    lemmatizer = WordNetLemmatizer()

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return ' '.join(tokens)

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
corpus = [preprocess_text(text) for text in df.Email_text]

In [ ]:
# Data
X = corpus
y = df["Labels"]

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])


In [ ]:
param_grid = {
    "tfidf__max_features": [3000, 5000, 10000, None],
    "tfidf__ngram_range": [(1,1), (1,2), (2,2)],
    "tfidf__min_df": [1, 2, 5],
    "clf__C": [0.1, 1, 10],
    "clf__class_weight": [None, "balanced"]
}

In [ ]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

In [ ]:
grid.fit(X_train, y_train)

In [ ]:
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)
best_model_LG = grid.best_estimator_

In [ ]:
df['Labels'].value_counts()

**GRU**

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

texts = df["Email_text"].astype(str)
labels = df["Labels"]

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# Tokenizer
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# Convert texts to sequences
train_seq = tokenizer.texts_to_sequences(X_train)
test_seq = tokenizer.texts_to_sequences(X_test)

# Padding
max_len = 100
X_train_pad = pad_sequences(train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(test_seq, maxlen=max_len, padding='post')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

model = Sequential([
    Embedding(input_dim=10000, output_dim=128),
    GRU(64),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
pip install keras_tuner


In [ ]:
import keras_tuner as kt
from tensorflow.keras.layers import GRU, Embedding, Dense, Dropout
from tensorflow.keras.models import Sequential

def build_model(hp):

    model = Sequential()

    model.add(
        Embedding(
            input_dim=10000,
            output_dim=hp.Choice('embed_dim', [64, 128, 256])
        )
    )

    model.add(
        GRU(
            units=hp.Choice('gru_units', [32, 64, 128]),
            return_sequences=False
        )
    )

    model.add(
        Dropout(hp.Choice('dropout', [0.2, 0.3, 0.5]))
    )

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    directory='gru_tuning',
    project_name='spam_gru'
)

In [ ]:
tuner.search(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=32
)

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
loss, acc = best_model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

**BERT**

In [ ]:
#!pip install transformers datasets torch -q

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = df[['Email_text', 'Labels']].dropna()

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Email_text'].astype(str),
    df['Labels'],
    test_size=0.2,
    random_state=42
)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
import torch

class SpamDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.values

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SpamDataset(train_encodings, train_labels)
test_dataset = SpamDataset(test_encodings, test_labels)

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
!pip install -U transformers

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
import pickle

# Sauvegarder
with open('BERT_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Modèle sauvegardé !")

In [ ]:
import pickle

# Sauvegarder
with open('GRU_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("Modèle sauvegardé !")

In [ ]:
import pickle

# Sauvegarder
with open('LG_model.pkl', 'wb') as f:
    pickle.dump(best_model_LG, f)

print("Modèle sauvegardé !")

In [ ]:
from google.colab import files

files.download('BERT_model.pkl')
files.download('LG_model.pkl')
files.download('GRU_model.pkl')